# The Hidden Network of Words
### Visualising how humans connect concepts — Scientific Data Visualisation project

**Data:** Small World of Words, English (SWOW-EN, 2018 release, `SWOW-EN.R100.csv`)
**Tools:** Python, networkx, scikit-learn, XGBoost, UMAP, matplotlib, plotly

**Notebook map**

1. Setup and visual style
2. Load data and **verify the structure**
3. Cleaning and exploratory analysis
4. Building the association network
5. Candidate research questions
6. Network analysis: communities (Q1) and bridge words (Q2)
7. Word embeddings and unsupervised learning (method comparison)
8. Supervised learning: predicting missing associations (Q3)
9. Multimodal extension with THINGS (optional)
10. Conclusions, limitations, presentation storyboard

> **Status:** this notebook was written without access to the real SWOW file, so every number and figure appears only after you run it on the real data. Section 2 prints the real structure first. Nothing below should be quoted in the presentation until you have seen it in your own output.

**How to get the data:** download `SWOW-EN.R100.csv` from the *Research* page of https://smallworldofwords.org (SWOW-EN, "Raw data") and place it in `data/raw/`. That folder is in `.gitignore`, so each teammate downloads it once.

## 1. Setup and visual style

All figures share one palette, so a community keeps the same colour in every chart. The palette is colour-blind safe (Okabe–Ito). Grey is reserved for "everything else", which keeps the colour focused on the groups being discussed.

Gestalt choices used throughout:

| Principle | How we apply it |
|---|---|
| **Similarity** | same community = same colour, in every figure |
| **Proximity** | layouts place related words close together (spring layout, UMAP) |
| **Connectedness** | lines are drawn only for strong associations, so a line means a real link |
| **Figure–ground** | context in light grey, focus in colour; no gridlines or boxes |
| **Continuity** | line charts for ordered variables (e.g. number of clusters) |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
import plotly.graph_objects as go
from scipy import sparse
from scipy.stats import spearmanr
import umap

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             normalized_mutual_info_score, roc_auc_score,
                             roc_curve, silhouette_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "raw" / "SWOW-EN.R100.csv"
FIG_DIR = ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MIN_COUNT = 2          # an association must be given by at least this many participants
FOCUS_WORD = "money"   # word used for the interactive ego network (falls back if absent)

PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00",
           "#CC79A7", "#56B4E9", "#8C564B", "#332288"]
GREY = "#BDBDBD"
INK = "#2B2B2B"
HALO = [pe.withStroke(linewidth=3, foreground="white")]   # keeps labels readable over points

sns.set_theme(style="white", context="notebook")
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK,
    "axes.titlesize": 13, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "figure.dpi": 110, "savefig.dpi": 200,
})

: 

## 2. Load data and verify the structure

We look at the file before using it. The cell below stops with a clear message if the columns we rely on (`cue`, `R1`, `R2`, `R3`) are not present. In SWOW-EN each row is one participant answering one cue word with up to three associations (`R1`, `R2`, `R3`).

In [ ]:
raw = pd.read_csv(DATA_FILE)
print("shape:", raw.shape)
display(raw.head())
display(raw.dtypes.to_frame("dtype").T)

needed = ["cue", "R1", "R2", "R3"]
assert all(c in raw.columns for c in needed), f"Expected columns {needed}, found {list(raw.columns)}"

## 3. Cleaning and exploratory analysis

Decisions made here:

* lower-case and strip whitespace, so `Dog` and `dog ` are one word;
* the placeholder `Unknown word` (participant did not know the cue) is treated as missing;
* a response identical to its cue is dropped;
* we **keep all three response positions**. R1 reflects the strongest association, while R2 and R3 add depth. This is a choice we can revisit.

In [ ]:
resp_cols = ["R1", "R2", "R3"]
for c in ["cue"] + resp_cols:
    raw[c] = raw[c].astype("string").str.lower().str.strip()

print("missing share per response column:")
print(raw[resp_cols].isna().mean().round(3).to_string())
unknown = raw[resp_cols].eq("unknown word").fillna(False)
print("\n'unknown word' share per response column:")
print(unknown.mean().round(3).to_string())

raw[resp_cols] = raw[resp_cols].mask(unknown)

long = (raw.melt(id_vars="cue", value_vars=resp_cols,
                 var_name="position", value_name="response")
           .dropna(subset=["response"]))
long = long[(long.response != long.cue) & (long.response != "")]

cues = set(raw.cue)
summary = pd.Series({
    "participant rows": len(raw),
    "distinct cues": len(cues),
    "valid responses": len(long),
    "distinct responses": long.response.nunique(),
    "share of responses that are themselves cues": long.response.isin(cues).mean().round(3),
    "median valid responses per cue": long.groupby("cue").size().median(),
})
display(summary.to_frame("value"))

In [ ]:
resp_counts = long.response.value_counts()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))

top = resp_counts.head(15)[::-1]
ax[0].barh(top.index, top.values, color=PALETTE[0])
ax[0].set_title("The 15 most common responses")
ax[0].set_xlabel("times given")

ax[1].loglog(np.arange(1, len(resp_counts) + 1), resp_counts.values, color=PALETTE[0], lw=2)
ax[1].set_title("Few words absorb most responses")
ax[1].set_xlabel("response rank (log)")
ax[1].set_ylabel("times given (log)")

fig.tight_layout()
fig.savefig(FIG_DIR / "01_eda_responses.png", bbox_inches="tight")
plt.show()

top100_share = resp_counts.head(100).sum() / resp_counts.sum()
print(f"The 100 most common responses account for {top100_share:.1%} of all responses.")

## 4. Building the association network

**Nodes** are cue words. A response becomes a node only if it is also a cue; this is the usual SWOW setup, and it avoids a huge tail of one-off words.
**Edges** are directed, `cue → response`. Edge strength is the share of the cue's valid responses that were this word.

Two filters keep the network meaningful:

* `MIN_COUNT`: an edge needs at least this many participants, which removes idiosyncratic answers;
* we keep the largest connected component.

For community detection and link prediction we also use an **undirected** version where the weight is the sum of both directions.

In [ ]:
edges = long.groupby(["cue", "response"]).size().rename("count").reset_index()
edges["strength"] = edges["count"] / edges.groupby("cue")["count"].transform("sum")

net_edges = edges[edges.response.isin(cues) & (edges["count"] >= MIN_COUNT)]

DG = nx.from_pandas_edgelist(net_edges, "cue", "response",
                             edge_attr=["strength", "count"], create_using=nx.DiGraph)
lcc = max(nx.weakly_connected_components(DG), key=len)
DG = DG.subgraph(lcc).copy()

pair_w = {}
for u, v, d in DG.edges(data=True):
    key = tuple(sorted((u, v)))
    pair_w[key] = pair_w.get(key, 0) + d["strength"]
UG = nx.Graph()
UG.add_weighted_edges_from([(u, v, w) for (u, v), w in pair_w.items()])
nx.set_edge_attributes(UG, {(u, v): 1 / w for (u, v), w in pair_w.items()}, "dist")

print(f"directed:   {DG.number_of_nodes():,} nodes, {DG.number_of_edges():,} edges")
print(f"undirected: {UG.number_of_nodes():,} nodes, {UG.number_of_edges():,} edges")
print(f"density (undirected): {nx.density(UG):.5f}")
print(f"reciprocity (share of directed edges with a return edge): {nx.reciprocity(DG):.3f}")
print(f"cues kept: {UG.number_of_nodes() / len(cues):.1%}")

## 5. Candidate research questions

These are starting points. Revise them after looking at the outputs below, and keep the 2–3 most interesting ones for the presentation.

* **Q1, organisation.** Does the association network split into communities that are stronger than chance (compared with degree-preserving random rewiring), and are those communities semantically coherent?
* **Q2, bridges.** Are the words that connect communities (high *participation coefficient*, high *betweenness*) different from the most connected words (hubs)?
* **Q3, predictability.** Can a held-out association between two words be predicted from network structure and word embeddings, and which model family works best? (No Free Lunch: we compare several instead of assuming one wins.)
* **Q4, multimodal (optional).** Do words for visually similar objects sit close together in association space? (Section 9, needs THINGS.)

## 6. Network analysis: communities (Q1) and bridge words (Q2)

Per-word metrics:

* `times_given`: how often the word was produced as a response (popularity);
* `degree`: number of neighbours in the undirected network;
* `pagerank`: importance from the directed, weighted network;
* `betweenness`: how often the word lies on shortest paths (approximated from a sample of start nodes, with strong links treated as short);
* `community` (Louvain, weighted) and `participation`, the extent to which a word's neighbours spread over several communities (0 = all neighbours in one community).

In [ ]:
nodes = list(UG.nodes())
nodes_df = pd.DataFrame(index=nodes)
nodes_df["times_given"] = long.response.value_counts().reindex(nodes).fillna(0).astype(int)
nodes_df["degree"] = pd.Series(dict(UG.degree()))
nodes_df["pagerank"] = pd.Series(nx.pagerank(DG, weight="strength"))
nodes_df["betweenness"] = pd.Series(
    nx.betweenness_centrality(UG, k=min(300, len(nodes)), weight="dist", seed=SEED))

comms = sorted(nx.community.louvain_communities(UG, weight="weight", seed=SEED),
               key=len, reverse=True)
nodes_df["community"] = pd.Series({w: k for k, c in enumerate(comms) for w in c})

A = nx.to_scipy_sparse_array(UG, nodelist=nodes, weight=None, format="csr")
codes, _ = pd.factorize(nodes_df.community)
onehot = sparse.csr_matrix((np.ones(len(nodes)), (np.arange(len(nodes)), codes)))
k_ic = A @ onehot
k_i = np.asarray(A.sum(1)).ravel()
nodes_df["participation"] = 1 - np.asarray(k_ic.multiply(k_ic).sum(1)).ravel() / k_i**2

# one colour mapping for the whole notebook: 8 largest communities coloured, rest grey
comm_sizes = nodes_df.community.value_counts()
top_comms = comm_sizes.index[:8].tolist()
comm_color = {c: PALETTE[i] for i, c in enumerate(top_comms)}
nodes_df["color"] = nodes_df.community.map(comm_color).fillna(GREY)
comm_name = {c: ", ".join(nodes_df[nodes_df.community == c].nlargest(3, "pagerank").index)
             for c in comm_sizes.index}

print(f"{len(comms)} communities; the 8 largest hold {comm_sizes.head(8).sum() / len(nodes):.1%} of words")
display(pd.DataFrame({"size": comm_sizes.head(10),
                      "top words (by PageRank)": [comm_name[c] for c in comm_sizes.head(10).index]}))
display(nodes_df.drop(columns="color").describe().T.round(4))

### Q1: are the communities stronger than chance?

We compare the modularity of the real network with networks of the same degree sequence but randomly rewired edges. To keep the comparison fair, both use unweighted edges.

In [ ]:
N_NULL = 3
U0 = nx.Graph(UG.edges())
q_real = nx.community.modularity(U0, nx.community.louvain_communities(U0, seed=SEED))

q_null = []
for s in range(N_NULL):
    H = nx.Graph(UG.edges())
    m = H.number_of_edges()
    nx.double_edge_swap(H, nswap=2 * m, max_tries=50 * m, seed=SEED + s)
    q_null.append(nx.community.modularity(H, nx.community.louvain_communities(H, seed=SEED)))

display(pd.Series({"modularity, real network": q_real,
                   "modularity, rewired (mean)": np.mean(q_null),
                   "modularity, rewired (min–max)": f"{min(q_null):.3f}–{max(q_null):.3f}"}).to_frame("value"))

### Visual 1: how the communities relate to each other

A word-level network of 12k nodes is an unreadable hairball. We first show the **community level**: one node per community (size = number of words, label = its three most central words), and lines only between communities that are strongly linked, relative to their size.

In [ ]:
show = top_comms
between = {}
for u, v, w in UG.edges(data="weight"):
    a, b = nodes_df.community[u], nodes_df.community[v]
    if a != b and a in show and b in show:
        key = tuple(sorted((a, b)))
        between[key] = between.get(key, 0) + w

M = nx.Graph()
M.add_nodes_from(show)
for (a, b), w in between.items():
    M.add_edge(a, b, w=w / np.sqrt(comm_sizes[a] * comm_sizes[b]))

pos = nx.spring_layout(M, weight="w", seed=SEED, k=1.2)
w_all = np.array([d["w"] for _, _, d in M.edges(data=True)])
w_cut = np.quantile(w_all, 0.5) if len(w_all) else 0

fig, ax = plt.subplots(figsize=(9, 7))
for a, b, d in M.edges(data=True):
    if d["w"] >= w_cut:
        ax.plot(*zip(pos[a], pos[b]), color=GREY, alpha=0.7, zorder=1,
                lw=1 + 6 * (d["w"] - w_all.min()) / (w_all.max() - w_all.min() + 1e-12))
for c in show:
    size = 250 + 2500 * comm_sizes[c] / comm_sizes[show].max()
    ax.scatter(*pos[c], s=size, color=comm_color[c], zorder=2, edgecolor="white", linewidth=1.5)
    ax.annotate(comm_name[c], pos[c], xytext=(0, -12 - np.sqrt(size) * 0.55),
                textcoords="offset points", ha="center", fontsize=9, path_effects=HALO)
ax.set_title("Communities of words and the strongest links between them")
ax.margins(0.18)
ax.axis("off")
fig.savefig(FIG_DIR / "02_community_map.png", bbox_inches="tight")
plt.show()

### Visual 2: inside the communities

We sample the 25 most central words from each of the six largest communities and draw only their strongest mutual links. Colour = community (similarity), position from the weighted layout (proximity).

In [ ]:
six = top_comms[:6]
sample_words = []
for c in six:
    sample_words += nodes_df[nodes_df.community == c].nlargest(25, "pagerank").index.tolist()

S = UG.subgraph(sample_words).copy()
cut = np.quantile([d["weight"] for _, _, d in S.edges(data=True)], 0.6)
S.remove_edges_from([(u, v) for u, v, d in S.edges(data=True) if d["weight"] < cut])
pos = nx.spring_layout(S, weight="weight", seed=SEED, k=0.35, iterations=200)

fig, ax = plt.subplots(figsize=(11, 8))
nx.draw_networkx_edges(S, pos, ax=ax, edge_color=GREY, alpha=0.6,
                       width=[0.4 + 8 * d["weight"] for _, _, d in S.edges(data=True)])
nx.draw_networkx_nodes(S, pos, ax=ax, node_color=nodes_df.color[list(S.nodes)],
                       node_size=nodes_df.pagerank[list(S.nodes)] / nodes_df.pagerank.max() * 700 + 40,
                       edgecolors="white", linewidths=1)
for c in six:
    for rank, w in enumerate(nodes_df[nodes_df.community == c].nlargest(2, "pagerank").index):
        if w in S:
            ax.annotate(w, pos[w], xytext=(0, 16 if rank == 0 else -22), textcoords="offset points",
                        ha="center", fontsize=10, fontweight="bold", path_effects=HALO)
ax.set_title("Core vocabulary of the six largest communities")
ax.axis("off")
fig.savefig(FIG_DIR / "03_core_network.png", bbox_inches="tight")
plt.show()

### Visual 3 (interactive): the neighbourhood of one word

Hover over a node to see its community. The focus word is set in `FOCUS_WORD` (it falls back to the highest-PageRank word if it is not in the network).

In [ ]:
focus = FOCUS_WORD if FOCUS_WORD in UG else nodes_df.pagerank.idxmax()
nbrs = sorted(UG[focus].items(), key=lambda x: -x[1]["weight"])[:25]
ego_nodes = [focus] + [n for n, _ in nbrs]
H = UG.subgraph(ego_nodes)
pos = nx.spring_layout(H, weight="weight", seed=SEED)

ex, ey = [], []
for u, v in H.edges():
    ex += [pos[u][0], pos[v][0], None]
    ey += [pos[u][1], pos[v][1], None]

fig = go.Figure([
    go.Scatter(x=ex, y=ey, mode="lines", line=dict(width=0.7, color=GREY), hoverinfo="skip"),
    go.Scatter(x=[pos[n][0] for n in ego_nodes], y=[pos[n][1] for n in ego_nodes],
               mode="markers+text", text=ego_nodes, textposition="top center",
               marker=dict(size=[34 if n == focus else 14 + 2 * np.sqrt(H.degree(n)) for n in ego_nodes],
                           color=nodes_df.color[ego_nodes], line=dict(width=1.5, color="white")),
               hovertext=[f"{n}<br>community: {comm_name[nodes_df.community[n]]}" for n in ego_nodes],
               hoverinfo="text"),
])
fig.update_layout(template="simple_white", showlegend=False, height=600, width=800,
                  title=f"Strongest associations of '{focus}'",
                  xaxis=dict(visible=False), yaxis=dict(visible=False))
fig.write_html(FIG_DIR / "04_ego_network.html")
fig.show()

### Q2: hubs versus bridges

A **hub** has many neighbours. A **bridge** has neighbours in many different communities. If the two roles were the same, the points would lie on a clear diagonal. We check this with rank correlations and list the strongest bridges by betweenness.

In [ ]:
rho_deg_part = spearmanr(nodes_df.degree, nodes_df.participation)[0]
rho_deg_btw = spearmanr(nodes_df.degree, nodes_df.betweenness)[0]
print(f"Spearman degree vs participation: {rho_deg_part:.2f}")
print(f"Spearman degree vs betweenness:   {rho_deg_btw:.2f}")

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(nodes_df.degree, nodes_df.participation, s=10, c=nodes_df.color, alpha=0.7, linewidths=0)
for w in nodes_df.nlargest(10, "betweenness").index:
    ax.annotate(w, (nodes_df.degree[w], nodes_df.participation[w]),
                xytext=(5, 4), textcoords="offset points", fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("degree (number of neighbours, log scale)")
ax.set_ylabel("participation coefficient")
ax.set_title("Connected is not the same as connecting")
fig.savefig(FIG_DIR / "05_hubs_vs_bridges.png", bbox_inches="tight")
plt.show()

bridges = nodes_df.nlargest(15, "betweenness")[["degree", "participation", "betweenness"]].copy()
bridges["community"] = [comm_name[nodes_df.community[w]] for w in bridges.index]
display(bridges.round(3))

## 7. Word embeddings and unsupervised learning

**Embedding.** We turn the weighted network into word vectors with the classic *PPMI + SVD* recipe: positive pointwise mutual information of the association matrix, reduced to 100 dimensions. Words that share many associates end up close together (cosine similarity).

**Clustering, a No Free Lunch comparison.** Three methods that make different assumptions:

* **Louvain**: graph-based, maximises modularity (already computed);
* **K-means**: spherical clusters in embedding space;
* **Ward agglomerative clustering**: hierarchical, minimises within-cluster variance.

K-means and Ward use the same number of clusters as Louvain has substantial communities (≥ 30 words). Each method is judged by two criteria. *Modularity* is the graph view and favours Louvain by construction. *Silhouette* is the embedding view and favours K-means-like methods. Neither view is neutral, which is the point of the comparison.

In [ ]:
def ppmi_svd(W, dim=100, seed=SEED):
    W = sparse.csr_matrix(W)
    total = W.sum()
    row_sum = np.asarray(W.sum(1)).ravel()
    row_sum[row_sum == 0] = 1
    c = W.tocoo()
    pmi = np.log(c.data * total / (row_sum[c.row] * row_sum[c.col]))
    ppmi = sparse.csr_matrix((np.maximum(pmi, 0), (c.row, c.col)), shape=W.shape)
    ppmi.eliminate_zeros()
    svd = TruncatedSVD(min(dim, W.shape[0] - 1), random_state=seed)
    return normalize(svd.fit_transform(ppmi))

idx = {w: i for i, w in enumerate(nodes)}
W_full = nx.to_scipy_sparse_array(UG, nodelist=nodes, weight="weight", format="csr")
emb = ppmi_svd(W_full)

# sanity check: nearest neighbours in embedding space for a few words
for w in [focus] + nodes_df.nlargest(2, "pagerank").index.tolist():
    sims = emb @ emb[idx[w]]
    near = [nodes[i] for i in np.argsort(-sims)[1:9]]
    print(f"{w:>12}: {', '.join(near)}")

In [ ]:
K_eff = int((comm_sizes >= 30).sum())
labels = {"Louvain (graph)": nodes_df.community.to_numpy(),
          "K-means (embedding)": KMeans(K_eff, n_init=5, random_state=SEED).fit_predict(emb),
          "Ward (embedding)": AgglomerativeClustering(K_eff, linkage="ward").fit_predict(emb)}

rows = []
for name, lab in labels.items():
    groups = [set(np.array(nodes)[lab == k]) for k in np.unique(lab)]
    rows.append({"method": name, "clusters": len(groups),
                 "modularity (graph view)": nx.community.modularity(UG, groups, weight="weight"),
                 "silhouette (embedding view)": silhouette_score(emb, lab, metric="cosine",
                                                                 sample_size=min(3000, len(nodes)),
                                                                 random_state=SEED)})
method_table = pd.DataFrame(rows).set_index("method")
display(method_table.round(3))

names = list(labels)
agree = pd.DataFrame([[adjusted_rand_score(labels[a], labels[b]) for b in names] for a in names],
                     index=names, columns=names)
fig, ax = plt.subplots(figsize=(5.6, 4.4))
sns.heatmap(agree, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax)
ax.set_title("Do the methods find the same groups?\n(adjusted Rand index)", fontsize=11)
fig.savefig(FIG_DIR / "06_method_agreement.png", bbox_inches="tight")
plt.show()

In [ ]:
ks = [4, 8, 12, 16, 24, 32, 48]
sil = [silhouette_score(emb, KMeans(k, n_init=3, random_state=SEED).fit_predict(emb), metric="cosine",
                        sample_size=min(3000, len(nodes)), random_state=SEED) for k in ks]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(ks, sil, marker="o", color=PALETTE[0], lw=2)
ax.set_xlabel("number of K-means clusters")
ax.set_ylabel("silhouette")
ax.set_title("Is there a natural number of clusters?")
fig.savefig(FIG_DIR / "07_kmeans_sweep.png", bbox_inches="tight")
plt.show()

### Visual 4: a map of the mental lexicon

UMAP projects the embeddings to 2D (proximity: close points have similar associations). Colour is the Louvain community, so the **graph-based** groups are being compared with the **embedding-based** geometry. If colours form compact regions, the two views agree. Each of the eight largest communities is labelled with its most central words.

In [ ]:
xy = umap.UMAP(n_neighbors=30, min_dist=0.15, metric="cosine", random_state=SEED).fit_transform(emb)
nodes_df["x"], nodes_df["y"] = xy[:, 0], xy[:, 1]

fig, ax = plt.subplots(figsize=(11, 8))
grey = nodes_df[~nodes_df.community.isin(top_comms)]
ax.scatter(grey.x, grey.y, s=4, color=GREY, alpha=0.5, linewidths=0)
for c in top_comms:
    d = nodes_df[nodes_df.community == c]
    ax.scatter(d.x, d.y, s=5, color=comm_color[c], alpha=0.8, linewidths=0)
    ax.annotate(comm_name[c], (d.x.median(), d.y.quantile(0.97)), xytext=(0, 8),
                textcoords="offset points", ha="center", fontsize=10, fontweight="bold",
                color=INK, path_effects=HALO)
ax.set_title("A map of English word associations")
ax.axis("off")
fig.savefig(FIG_DIR / "08_umap_map.png", bbox_inches="tight")
plt.show()

In [ ]:
fig = go.Figure()
other = nodes_df[~nodes_df.community.isin(top_comms)]
fig.add_trace(go.Scattergl(x=other.x, y=other.y, mode="markers", name="other communities",
                           marker=dict(size=3, color=GREY, opacity=0.5),
                           text=other.index, hoverinfo="text"))
for c in top_comms:
    d = nodes_df[nodes_df.community == c]
    fig.add_trace(go.Scattergl(x=d.x, y=d.y, mode="markers", name=comm_name[c],
                               marker=dict(size=4, color=comm_color[c], opacity=0.85),
                               text=d.index, hoverinfo="text"))
fig.update_layout(template="simple_white", height=700, width=1000,
                  title="Interactive map of word associations (click legend to toggle communities)",
                  xaxis=dict(visible=False), yaxis=dict(visible=False))
fig.write_html(FIG_DIR / "09_umap_interactive.html")
fig.show()

## 8. Supervised learning: predicting missing associations (Q3)

**Task.** Given two words, predict whether they are linked in the network. Positive pairs are real edges, and negative pairs are random non-edges.

**No leakage.** All pairs are split first: 70% of the edges form the *feature graph*, 15% are *training labels*, and 15% are *test labels*. Features for training and test pairs (neighbour overlap, embeddings, communities) are computed **only from the feature graph**, so a pair's own edge can never leak into its features.

**Features:** common neighbours, Jaccard, Adamic–Adar, degree sum (log), cosine similarity of the PPMI-SVD embedding, and same-community.

**Models (No Free Lunch):** logistic regression, random forest and XGBoost. Single-feature baselines are shown for reference.

> **Caveat.** With random negatives this task is easier than real "predict the missing link" problems. Treat the absolute scores as an upper bound and focus on the *differences* between models and feature groups.

In [ ]:
rng = np.random.default_rng(SEED)

ed = nx.to_pandas_edgelist(UG)
E = pd.DataFrame({"i": ed.source.map(idx), "j": ed.target.map(idx), "w": ed.weight})
order = rng.permutation(len(E))
n_e = len(E)
E_graph = E.iloc[order[:int(.70 * n_e)]]
E_train = E.iloc[order[int(.70 * n_e):int(.85 * n_e)]][["i", "j"]].to_numpy()
E_test = E.iloc[order[int(.85 * n_e):]][["i", "j"]].to_numpy()

N = len(nodes)
edge_keys = set((np.minimum(E.i, E.j) * N + np.maximum(E.i, E.j)).tolist())

def sample_negatives(n):
    out = []
    while len(out) < n:
        a, b = rng.integers(0, N, 2)
        if a != b and min(a, b) * N + max(a, b) not in edge_keys:
            out.append((a, b))
            edge_keys.add(min(a, b) * N + max(a, b))
    return np.array(out)

neg_train = sample_negatives(len(E_train))
neg_test = sample_negatives(len(E_test))

# everything below uses the feature graph only
Wg = sparse.coo_matrix((E_graph.w, (E_graph.i, E_graph.j)), shape=(N, N))
Wg = (Wg + Wg.T).tocsr()
Ag = (Wg > 0).astype(float).tocsr()
deg = np.asarray(Ag.sum(1)).ravel()
aa_w = np.where(deg > 1, 1 / np.log(np.maximum(deg, 2)), 0)
Ag_w = Ag @ sparse.diags(aa_w)
emb_g = ppmi_svd(Wg)

Gg = nx.Graph()
Gg.add_nodes_from(range(N))
Gg.add_weighted_edges_from(zip(E_graph.i, E_graph.j, E_graph.w))
comm_g = np.zeros(N, dtype=int)
for k, c in enumerate(nx.community.louvain_communities(Gg, weight="weight", seed=SEED)):
    comm_g[list(c)] = k

def pair_features(pairs):
    i, j = pairs[:, 0], pairs[:, 1]
    cn = np.asarray(Ag[i].multiply(Ag[j]).sum(1)).ravel()
    aa = np.asarray(Ag_w[i].multiply(Ag[j]).sum(1)).ravel()
    return pd.DataFrame({
        "common_neighbours": cn,
        "jaccard": cn / np.maximum(deg[i] + deg[j] - cn, 1),
        "adamic_adar": aa,
        "log_degree_sum": np.log1p(deg[i]) + np.log1p(deg[j]),
        "embedding_cosine": (emb_g[i] * emb_g[j]).sum(1),
        "same_community": (comm_g[i] == comm_g[j]).astype(int)})

X_train = pair_features(np.vstack([E_train, neg_train]))
y_train = np.r_[np.ones(len(E_train)), np.zeros(len(neg_train))]
X_test = pair_features(np.vstack([E_test, neg_test]))
y_test = np.r_[np.ones(len(E_test)), np.zeros(len(neg_test))]
print("train pairs:", X_train.shape, " test pairs:", X_test.shape)

In [ ]:
models = {
    "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random forest": RandomForestClassifier(300, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
                             eval_metric="logloss", n_jobs=-1, random_state=SEED),
}
scores, probs = {}, {}
for name, mdl in models.items():
    mdl.fit(X_train, y_train)
    probs[name] = mdl.predict_proba(X_test)[:, 1]
    scores[name] = {"ROC-AUC": roc_auc_score(y_test, probs[name]),
                    "Average precision": average_precision_score(y_test, probs[name])}
for f in ["embedding_cosine", "common_neighbours"]:
    scores[f"(baseline) {f}"] = {"ROC-AUC": roc_auc_score(y_test, X_test[f]),
                                 "Average precision": average_precision_score(y_test, X_test[f])}
results = pd.DataFrame(scores).T
display(results.round(3))
best = results.loc[list(models)]["ROC-AUC"].idxmax()
print("best model on ROC-AUC:", best)

In [ ]:
model_colors = dict(zip(models, PALETTE[:3]))
groups = {"local structure only": ["common_neighbours", "jaccard", "adamic_adar", "log_degree_sum"],
          "embedding only": ["embedding_cosine"],
          "community only": ["same_community"],
          "all features": list(X_train.columns)}
ablation = {}
for g, cols in groups.items():
    m = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
                      eval_metric="logloss", n_jobs=-1, random_state=SEED).fit(X_train[cols], y_train)
    ablation[g] = roc_auc_score(y_test, m.predict_proba(X_test[cols])[:, 1])
ablation = pd.Series(ablation)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.8))
for name in models:
    fpr, tpr, _ = roc_curve(y_test, probs[name])
    ax[0].plot(fpr, tpr, color=model_colors[name], lw=2.2,
               label=f"{name} ({scores[name]['ROC-AUC']:.3f})")
for f, ls in [("embedding_cosine", "--"), ("common_neighbours", ":")]:
    fpr, tpr, _ = roc_curve(y_test, X_test[f])
    ax[0].plot(fpr, tpr, color=GREY, lw=1.8, ls=ls, label=f"baseline: {f}")
ax[0].plot([0, 1], [0, 1], color="#E0E0E0", lw=1)
ax[0].set_xlabel("false positive rate")
ax[0].set_ylabel("true positive rate")
ax[0].set_title("Three model families, one task")
ax[0].legend(frameon=False, fontsize=9, loc="lower right")

ab = ablation.sort_values()
cols = [PALETTE[0] if g == "all features" else GREY for g in ab.index]
ax[1].hlines(ab.index, 0.5, ab.values, color=cols, lw=3)
ax[1].scatter(ab.values, ab.index, color=cols, s=90, zorder=3)
ax[1].axvline(0.5, color=INK, lw=0.8)
ax[1].set_xlim(0.45, 1.02)
ax[1].set_xlabel("ROC-AUC (XGBoost); lines start at chance level 0.5")
ax[1].set_title("Which information matters?")
for y, v in enumerate(ab.values):
    ax[1].text(v + 0.012, y, f"{v:.3f}", va="center", fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "10_link_prediction.png", bbox_inches="tight")
plt.show()

### What drives the best model, and which "missing" links does it find?

Permutation importance shows how much the test ROC-AUC drops when a feature is shuffled. Then we list the highest-scoring pairs among test **non-edges**. These are word pairs the model thinks *should* be associated even though they are not linked in our network. For each, we look up the raw count in the unfiltered data (either direction); a small positive count means the association exists but fell below `MIN_COUNT`, which supports the model's guess.

In [ ]:
sample = rng.choice(len(X_test), size=min(20000, len(X_test)), replace=False)
perm = permutation_importance(models[best], X_test.iloc[sample], y_test[sample],
                              scoring="roc_auc", n_repeats=5, random_state=SEED)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.barh(imp.index, imp.values, color=model_colors[best])
ax.set_xlabel("drop in ROC-AUC when shuffled")
ax.set_title(f"Feature importance ({best})")
fig.savefig(FIG_DIR / "11_feature_importance.png", bbox_inches="tight")
plt.show()

raw_count = edges.set_index(["cue", "response"])["count"].to_dict()
neg_scores = pd.DataFrame(neg_test, columns=["i", "j"])
neg_scores["score"] = probs[best][len(E_test):]
top_neg = neg_scores.nlargest(15, "score").copy()
top_neg["word A"] = [nodes[i] for i in top_neg.i]
top_neg["word B"] = [nodes[j] for j in top_neg.j]
top_neg["raw count A→B / B→A"] = [
    f"{raw_count.get((a, b), 0)} / {raw_count.get((b, a), 0)}"
    for a, b in zip(top_neg["word A"], top_neg["word B"])]
top_neg["same community"] = [comm_g[i] == comm_g[j] for i, j in zip(top_neg.i, top_neg.j)]
display(top_neg[["word A", "word B", "score", "raw count A→B / B→A", "same community"]].round(3).reset_index(drop=True))

## 9. Multimodal extension with THINGS (optional)

The THINGS database links ~1,850 object concepts to images and metadata. The question: **do words for related visual objects sit close together in association space?** This section runs only if you provide the files. It is guarded, so the notebook works without it.

**What to do first:**
1. Download the THINGS concept list (and optionally per-concept image embeddings) from https://things-initiative.org and save them in `data/raw/`.
2. Set the paths below and run the next cell. It **prints the real column names**. Set `THINGS_WORD_COL` and `THINGS_CATEGORY_COL` to match what you see; the defaults are guesses.
3. Optional: `THINGS_IMG_EMB_FILE` is a `.npy` array with one row per concept, in the same row order as the concept file. It enables the similarity comparison and the fusion experiment.

In [ ]:
THINGS_FILE = None            # e.g. ROOT / "data" / "raw" / "things_concepts.tsv"
THINGS_WORD_COL = "Word"
THINGS_CATEGORY_COL = None    # set after inspecting the printed columns
THINGS_IMG_EMB_FILE = None    # e.g. ROOT / "data" / "raw" / "things_image_embeddings.npy"

has_things = False
if THINGS_FILE is not None and Path(THINGS_FILE).exists():
    things = pd.read_csv(THINGS_FILE, sep=None, engine="python")
    print("columns:", list(things.columns))
    display(things.head())
    if THINGS_CATEGORY_COL in things.columns and THINGS_WORD_COL in things.columns:
        things["word_l"] = things[THINGS_WORD_COL].astype("string").str.lower().str.strip()
        things["row"] = np.arange(len(things))
        m = things[things.word_l.isin(idx) & things[THINGS_CATEGORY_COL].notna()].copy()
        keep = m[THINGS_CATEGORY_COL].value_counts()
        m = m[m[THINGS_CATEGORY_COL].isin(keep[keep >= 10].index)]
        has_things = len(m) > 50
        print(f"{len(m)} THINGS concepts matched to SWOW, {m[THINGS_CATEGORY_COL].nunique()} categories with >= 10 members")
    else:
        print("Set THINGS_WORD_COL and THINGS_CATEGORY_COL to column names from the list above.")
else:
    print("THINGS not provided: section 9 is skipped.")

In [ ]:
if has_things:
    Em = emb[[idx[w] for w in m.word_l]]
    cat = m[THINGS_CATEGORY_COL].to_numpy()
    sim = Em @ Em.T
    iu = np.triu_indices(len(m), 1)
    same = (cat[:, None] == cat[None, :])[iu]
    s = sim[iu]

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.kdeplot(s[~same], ax=ax, color=GREY, fill=True, label="different category", lw=1.5)
    sns.kdeplot(s[same], ax=ax, color=PALETTE[0], fill=True, label="same category", lw=1.5)
    ax.set_xlabel("cosine similarity in association space")
    ax.set_title("Do associations know object categories?")
    ax.legend(frameon=False)
    fig.savefig(FIG_DIR / "12_things_category_similarity.png", bbox_inches="tight")
    plt.show()
    print(f"mean similarity: same category {s[same].mean():.3f}, different {s[~same].mean():.3f}")

    nmi = normalized_mutual_info_score(cat, nodes_df.community.loc[m.word_l].to_numpy())
    print(f"NMI between THINGS category and Louvain community: {nmi:.3f}")

In [ ]:
if has_things:
    from sklearn.model_selection import cross_val_predict, StratifiedKFold
    from sklearn.metrics import accuracy_score, f1_score

    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    clf = LogisticRegression(max_iter=2000)
    pred = cross_val_predict(clf, Em, cat, cv=skf)
    majority = pd.Series(cat).value_counts(normalize=True).iloc[0]
    print(f"category prediction from association embedding (5-fold CV): "
          f"accuracy {accuracy_score(cat, pred):.3f}, macro-F1 {f1_score(cat, pred, average='macro'):.3f} "
          f"(majority-class accuracy {majority:.3f})")

In [ ]:
if has_things and THINGS_IMG_EMB_FILE is not None and Path(THINGS_IMG_EMB_FILE).exists():
    img = np.load(THINGS_IMG_EMB_FILE)
    assert img.shape[0] == len(things), "image embeddings must have one row per concept, in file order"
    Im = normalize(img[m.row.to_numpy()])

    # 1) do the two modalities agree on which concepts are similar?
    rho = spearmanr(sim[iu], (Im @ Im.T)[iu])[0]
    print(f"Spearman correlation of pairwise similarities, language vs image: {rho:.3f}")

    # 2) early fusion: concatenate both normalised embeddings, cluster, compare with categories
    k = len(np.unique(cat))
    views = {"language (SWOW)": Em, "image (THINGS)": Im, "fused": np.hstack([Em, Im])}
    fusion = {n: normalized_mutual_info_score(cat, KMeans(k, n_init=5, random_state=SEED).fit_predict(v))
              for n, v in views.items()}
    fusion = pd.Series(fusion)

    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    ax.bar(fusion.index, fusion.values, color=[GREY, GREY, PALETTE[0]])
    ax.set_ylabel("NMI with THINGS category")
    ax.set_title("Does combining modalities help?")
    fig.savefig(FIG_DIR / "13_things_fusion.png", bbox_inches="tight")
    plt.show()
else:
    print("Image embeddings not provided: similarity comparison and fusion experiment skipped.")

## 10. Conclusions, limitations, storyboard

### Conclusions (fill in after running; use numbers from your own outputs)

* **Q1:** real modularity vs. rewired modularity: ___ vs ___. Are the largest communities coherent? Quote 2–3 examples from the community table.
* **Q2:** Spearman(degree, participation) = ___. Name 3–4 bridge words from the table and say what communities they connect.
* **Q3:** best model ___, ROC-AUC ___. Which feature group carried most of the signal (ablation)? Which top "false positives" had a non-zero raw count?
* **Method comparison (NFL):** which method won on which criterion (modularity vs silhouette), and what the agreement heatmap says.
* **Multimodal:** if THINGS was used, the category-similarity gap and the fusion result.

### Limitations to state honestly

* SWOW is a snapshot of responses from a self-selected online sample and is not representative of all English speakers.
* Restricting nodes to cue words and requiring `MIN_COUNT` ≥ 2 removes rare associations and idiosyncratic knowledge.
* Louvain is stochastic and resolution-dependent; the number of communities is not a fact about language.
* Link prediction uses random negatives. Absolute scores are optimistic, and a harder variant (negatives with shared neighbours) is a natural extension.
* PPMI-SVD embeddings are one representation. Comparing them with pretrained text embeddings would be a good robustness check.

### Suggested 20-minute storyboard

| min | Part | Figure |
|---|---|---|
| 0–2 | Hook: one word, its neighbourhood (interactive ego network) | `04_ego_network.html` |
| 2–5 | Data and method: SWOW, how the network is built, what we filtered | `01_eda_responses.png` |
| 5–9 | Q1: the map of the lexicon | `08_umap_map.png`, `02_community_map.png` |
| 9–12 | Q2: hubs vs bridges | `05_hubs_vs_bridges.png` |
| 12–16 | Q3 and NFL: model comparison, what matters, surprising links | `10_link_prediction.png`, `11_feature_importance.png` |
| 16–18 | Multimodal extension (if done) | `12_*`, `13_*` |
| 18–20 | Conclusions and limitations | – |